# Compare FFT before/after 60/100 Hz notch removal

Loads the same sample(s) from the original dataset (`SRC_DIR`) and the new denotched copy
(`DST_DIR`, built by `build_denotched_dataset.py`), and plots magnitude spectra before/after
the notch, both unnormalized and normalized (`normalize_fft` from `post_process.py`).

In [ ]:
from pathlib import Path\n\nimport numpy as np\nimport matplotlib.pyplot as plt\n\nfrom io_utils import load\nfrom post_process import extract_signal, normalize_fft\nfrom denotch import interpolate_notch\n\n# **** point these at your actual dataset ****\nSRC_DIR = Path(r\"D:/eturok/experiment-22/data/samples\")  # original\nDST_DIR = Path(r\"D:/eturok/experiment-22-denotched/data/samples\")  # produced by build_denotched_dataset.py\nSAMPLE_IDS = None  # e.g. [\"sample_0000\", \"sample_0001\"]; None = first 3 found under SRC_DIR\n\nSPIKE_FREQS = [60.0, 100.0]\nNOTCH_HALF_WIDTH_HZ = 3.0\nSIGNAL_MODE = \"magnitude\"\nNORMALIZE_MODE = \"z-sample\"\n\nsample_ids = SAMPLE_IDS or sorted(p.name for p in SRC_DIR.glob(\"sample_*\"))[:3]\nprint(f\"Comparing {len(sample_ids)} samples: {sample_ids}\")"

In [ ]:
def load_sample_fft(sample_id: str):\n    \"\"\"Load raw complex fft + freqs from the ORIGINAL dataset (source of truth, always pre-denotch).\"\"\"\n    npz = load(SRC_DIR / sample_id / \"inputs/03_fft.npz\")\n    return npz[\"fft\"], npz[\"freqs\"]\n\n\ndef mean_magnitude(fft: np.ndarray) -> np.ndarray:\n    \"\"\"(B,L,F,2) complex -> (F,) magnitude averaged over batch, lasers, x/y.\"\"\"\n    mag = extract_signal(fft, \"magnitude\")  # (B,L,F,2)\n    return mag.mean(axis=(0, 1, 3))\n\n\nsamples = {}\nfor sid in sample_ids:\n    fft_before, freqs = load_sample_fft(sid)\n    fft_after = interpolate_notch(fft_before, freqs, spike_freqs=SPIKE_FREQS, half_width_hz=NOTCH_HALF_WIDTH_HZ)\n\n    mag_before_raw = extract_signal(fft_before, SIGNAL_MODE).astype(np.float32)\n    mag_after_raw = extract_signal(fft_after, SIGNAL_MODE).astype(np.float32)\n\n    mag_before_norm = normalize_fft(mag_before_raw, NORMALIZE_MODE)\n    mag_after_norm = normalize_fft(mag_after_raw, NORMALIZE_MODE)\n\n    samples[sid] = dict(\n        freqs=freqs,\n        before_unnorm=mean_magnitude(fft_before),\n        after_unnorm=mean_magnitude(fft_after),\n        before_norm=mag_before_norm.mean(axis=(0, 1, 3)),\n        after_norm=mag_after_norm.mean(axis=(0, 1, 3)),\n    )\nprint(\"Loaded + denotched:\", list(samples.keys()))"

## Full spectrum: unnormalized (top) vs normalized (bottom), before vs after

In [ ]:
for sid, d in samples.items():\n    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)\n    for ax, (before, after), title in [\n        (axes[0], (d[\"before_unnorm\"], d[\"after_unnorm\"]), \"unnormalized |fft|\"),\n        (axes[1], (d[\"before_norm\"], d[\"after_norm\"]), f\"normalized |fft| ({NORMALIZE_MODE})\"),\n    ]:\n        ax.plot(d[\"freqs\"], before, label=\"before\", alpha=0.8)\n        ax.plot(d[\"freqs\"], after, label=\"after\", alpha=0.8)\n        for f in SPIKE_FREQS: ax.axvline(f, color=\"red\", linestyle=\"--\", linewidth=0.8)\n        ax.set(title=f\"{sid}: {title}\", ylabel=\"magnitude\")\n        ax.legend()\n    axes[-1].set_xlabel(\"frequency (Hz)\")\n    fig.tight_layout()\n    plt.show()"

## Zoomed-in view around each spike (shows the interpolation directly)

In [ ]:
ZOOM_HALF_WINDOW_HZ = 15.0  # window shown on each side of a spike\n\nfor sid, d in samples.items():\n    fig, axes = plt.subplots(2, len(SPIKE_FREQS), figsize=(6 * len(SPIKE_FREQS), 7), sharex=\"col\")\n    freqs = d[\"freqs\"]\n    for col, spike_f in enumerate(SPIKE_FREQS):\n        zoom = (freqs >= spike_f - ZOOM_HALF_WINDOW_HZ) & (freqs <= spike_f + ZOOM_HALF_WINDOW_HZ)\n        for row, (before, after), title in [\n            (0, (d[\"before_unnorm\"], d[\"after_unnorm\"]), \"unnormalized\"),\n            (1, (d[\"before_norm\"], d[\"after_norm\"]), f\"normalized ({NORMALIZE_MODE})\"),\n        ]:\n            ax = axes[row, col]\n            ax.plot(freqs[zoom], before[zoom], \"o-\", label=\"before\", alpha=0.8, markersize=3)\n            ax.plot(freqs[zoom], after[zoom], \"o-\", label=\"after\", alpha=0.8, markersize=3)\n            ax.axvspan(spike_f - NOTCH_HALF_WIDTH_HZ, spike_f + NOTCH_HALF_WIDTH_HZ, color=\"red\", alpha=0.1)\n            ax.axvline(spike_f, color=\"red\", linestyle=\"--\", linewidth=0.8)\n            ax.set(title=f\"{sid}: {title} @ {spike_f} Hz\", xlabel=\"frequency (Hz)\")\n            if col == 0: ax.set_ylabel(\"magnitude\")\n            ax.legend()\n    fig.tight_layout()\n    plt.show()"